In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import copy
# --------------------------
# 基础模块 (CBR, Head, DownSample)
# --------------------------

class CBR(nn.Sequential):
    def __init__(self, ins, outs, k, s, p, d=1):
        super(CBR, self).__init__(
            nn.Conv2d(in_channels=ins, out_channels=outs, kernel_size=k, stride=s, padding=p, dilation=d, bias=False),
            nn.BatchNorm2d(num_features=outs),
            nn.ReLU(inplace=True)
        )

class DownSample(nn.Module):
    def __init__(self, ins, outs, k, s):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels=ins, out_channels=outs, kernel_size=k, stride=s, bias=False),
            nn.BatchNorm2d(num_features=outs)
        )
    def forward(self, x):
        return self.block(x)


# --------------------------
# 核心 Bottleneck
# --------------------------

class Bottleneck(nn.Module):
    def __init__(self, ins, outs, k, s, p, d, downsample=None):
        super().__init__()
        self.ds = downsample
        # 1x1 降维
        self.cbr1 = CBR(ins, outs, 1, 1, 0)
        # 3x3 处理 (注意这里的 dilation)
        self.cbr2 = CBR(outs, outs, 3, s, p, d)
        # 1x1 升维 (Expansion=4)
        self.cv = nn.Conv2d(outs, outs * 4, 1, 1, bias=False)
        self.bn = nn.BatchNorm2d(outs * 4)

    def forward(self, x):
        identity = x
        out = self.cbr1(x)
        out = self.cbr2(out)
        out = self.cv(out)
        out = self.bn(out)
        
        if self.ds is not None:
            identity = self.ds(x)
            
        out += identity
        return F.relu(out, inplace=True)

# --------------------------
# 各个 Layer 实现
# --------------------------

class Layer_1(nn.Module):
    def __init__(self, ins, outs, k=3, s=1, p=1, d=1):
        super().__init__()
        self.expansion = 4
        # Layer 1 输入通常与输出基数一致(64->64)，但Expansion后变256，所以需要Downsample
        ds = DownSample(ins, outs * self.expansion, 1, s)
        
        self.b1 = Bottleneck(ins, outs, k, s, p, d, downsample=ds)
        self.b2 = Bottleneck(outs * self.expansion, outs, k, 1, p, d)
        self.b3 = Bottleneck(outs * self.expansion, outs, k, 1, p, d)

    def forward(self, x):
        x = self.b1(x)
        x = self.b2(x)
        x = self.b3(x)
        return x

class Layer_2(nn.Module):
    def __init__(self, ins, outs): 
        # Ins: 256, Outs(Base): 128 -> Final: 512
        super().__init__()
        exp = 4
        self.b = nn.Sequential(
            # Stride=2 进行下采样
            Bottleneck(ins, outs, 3, 2, 1, 1, downsample=DownSample(ins, outs * exp, 1, 2)),
            Bottleneck(outs * exp, outs, 3, 1, 1, 1),
            Bottleneck(outs * exp, outs, 3, 1, 1, 1),
            Bottleneck(outs * exp, outs, 3, 1, 1, 1)
        )
    def forward(self, x):
        return self.b(x)

class Layer_3(nn.Module):
    def __init__(self, ins, outs):
        super().__init__()
        exp = 4
        
        self.b = nn.Sequential(
            Bottleneck(ins, outs, 3, 2, 1, 1, downsample=DownSample(ins, outs * exp, 1, 2)),
            Bottleneck(outs * exp, outs, 3, 1, 1, 1), 
            Bottleneck(outs * exp, outs, 3, 1, 1, 1),
            Bottleneck(outs * exp, outs, 3, 1, 1, 1),
            Bottleneck(outs * exp, outs, 3, 1, 1, 1),
            Bottleneck(outs * exp, outs, 3, 1, 1, 1),
        )
    def forward(self, x):
        return self.b(x)

class Layer_4(nn.Module):
    def __init__(self, ins, outs):
        super().__init__()
        exp = 4
        self.b = nn.Sequential(
            Bottleneck(ins, outs, 3, 2, 1, 1, downsample=DownSample(ins, outs * exp, 1, 2)),
            Bottleneck(outs * exp, outs, 3, 1, 1, 1), 
            Bottleneck(outs * exp, outs, 3, 1, 1, 1),
        )
    def forward(self, x):
        return self.b(x)

# --------------------------


class Encoder(nn.Module):
    def __init__(self, ins=3):
        super().__init__()
        
        # 1. Stem (前处理)
        # 输入: [B, 3, 480, 480] -> 输出: [B, 64, 120, 120]
        # cbr(ins, outs, k, s, p, d=1):
        
        self.cbr=CBR(ins, 64, 7, 2, 3)  # -> /2
            #k=3,s=2,p=1
        self.m=nn.MaxPool2d(3, 2, 1)   # -> /4
        
        
        # 2. Backbone (骨干)
        self.layer1 = Layer_1(ins=64, outs=64)
        
    
        self.layer2 = Layer_2(ins=256, outs=128)
        
        self.layer3 = Layer_3(ins=512, outs=256)
     
        self.layer4 = Layer_4(ins=1024, outs=512)
        
     

    def forward(self, x):
        input_size = x.shape[2:] # 记录 H, W: (224, 224)
        
        # --- 编码阶段 (Encoder) ---
        x = self.cbr(x)
        f1=x#跳跃链接
        x=self.m(x)
        x = self.layer1(x)
        f2=x#跳跃链接
        x = self.layer2(x)
        f3=x#跳跃链接
        x = self.layer3(x)
        f4=x#跳跃链接
        x = self.layer4(x)#Layer 4 Output: torch.Size([B, 2048, 16, 16])
        
        
        
        return x,[f1,f2,f3,f4]
    
    
class CCR(nn.Module):
    def __init__(self, ins, outs, k=3, s=1, p=1): # 注意 p=1 配合 k=3 才能保持尺寸
        super().__init__()
        self.b = nn.Sequential(
            # 第一层：负责把通道数降下来 (例如 3072 -> 512)
            nn.Conv2d(ins, outs, kernel_size=k, stride=s, padding=p, bias=False),
            nn.BatchNorm2d(outs), # 强烈建议加 BN
            nn.ReLU(inplace=True),
            
            # 第二层：输入已经是 outs 了！
            nn.Conv2d(outs, outs, kernel_size=k, stride=s, padding=p, bias=False),
            nn.BatchNorm2d(outs),
            nn.ReLU(inplace=True)
        ) 
    def forward(self, x):
        return self.b(x)


class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        """
        in_ch:   来自上一层(深层)的通道数 (例如 2048)
        skip_ch: 来自Encoder跳跃连接的通道数 (例如 1024)
        out_ch:  输出的目标通道数 (例如 512)
        """
        super().__init__()
        # 1. 上采样
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        
        # 2. 计算拼接后的总通道数 (自动计算，不用写死!)
        # 拼接后通道 = 上一层通道(in_ch) + 跳跃连接通道(skip_ch)
        concat_ch = in_ch + skip_ch 
        
        # 3. 定义卷积层
        self.ccr = CCR(ins=concat_ch, outs=out_ch)
        
    def forward(self, x, skip):
        # x: 来自深层
        # skip: 来自跳跃连接
        
        x = self.up(x)#上采样
        
        # 维度检查 (可选，防止尺寸由于取整问题不匹配)
        if x.size(2) != skip.size(2):
            x = F.interpolate(x, size=skip.shape[2:])
            
        x = torch.cat([skip, x], dim=1) # 在通道维拼接
        x = self.ccr(x)
        return x


class Decoder(nn.Module):
    def __init__(self, nums=64,filters=[64, 256, 512, 1024], bottom_ch=2048):
        """
        filters: Encoder 各层跳跃连接的通道数列表 [Layer1, Layer2, Layer3, Layer4_Skip]
                 对应 ResNet50: [64(Stem), 256, 512, 1024]
        bottom_ch: 最底层(Layer4 Output)的通道数，ResNet50 是 2048
        """
        super().__init__()
        # filters = [64, 256, 512, 1024]
        # 对应的 skip 顺序是从深到浅: f4(1024) -> f3(512) -> f2(256) -> f1(64)
        
        # Block 4: 融合 Bottom(2048) + Skip4(1024)
        # Out 设置为 filters[2] (512) 以便对接下一层
        self.dec4 = DecoderBlock(in_ch=bottom_ch, skip_ch=filters[3], out_ch=filters[2])
        
        # Block 3: 融合 Result(512) + Skip3(512)
        # Out 设置为 filters[1] (256)
        self.dec3 = DecoderBlock(in_ch=filters[2], skip_ch=filters[2], out_ch=filters[1])
        
        # Block 2: 融合 Result(256) + Skip2(256)
        # Out 设置为 128 (开始逐渐减小通道)
        self.dec2 = DecoderBlock(in_ch=filters[1], skip_ch=filters[1], out_ch=128)
        
        # Block 1: 融合 Result(128) + Skip1(64, Stem输出)
        # Out 设置为 64
        self.dec1 = DecoderBlock(in_ch=128, skip_ch=filters[0], out_ch=64)
        self.head = nn.Conv2d(filters[0],nums,kernel_size=1)
    def forward(self, x, skips):
        """
        x: Encoder 最后一层的输出 [B, 2048, H/32, W/32]
        skips: 跳跃连接列表 [f_stem, f_layer1, f_layer2, f_layer3]
               对应通道: [64, 256, 512, 1024]
               这里的是向量列表
        """
        # 1. 拆解 skips (注意顺序，f4是最深的，f1是最浅的)
        # 假设 skips 传进来是 [Stem, L1, L2, L3]
        f1, f2, f3, f4 = skips 
        
        # 2. 从下往上解码
        x = self.dec4(x, f4) # 融合 Layer4Out + Layer3
        x = self.dec3(x, f3) # 融合 + Layer2
        x = self.dec2(x, f2) # 融合 + Layer1
        x = self.dec1(x, f1) # 融合 + Stem
        #3.再次经过
        x=self.head(x)
        return x

class UNet(nn.Module):
    def __init__(self,ins=3,nums=2):
        super().__init__()
        self.ec = Encoder(ins)
        self.dc= Decoder(nums)
    def forward(self,x):
        #x 这里是原始尺寸[B 3 512 512]
        input_size = x.shape[2:]
        x,f = self.ec(x)
        x=self.dc(x,f)
        if x.size(2) != input_size[0] or x.size(3) != input_size[1]:
            x = F.interpolate(x, size=input_size, mode='bilinear', align_corners=True)
            
        return x

In [2]:
import os
import torch
import numpy as np
from torch.utils.data import Dataset,DataLoader
from PIL import Image
import torchvision.transforms as transforms

class CellDataset(Dataset):
    def __init__(self, root_dir, image_size=512, mode='train'):
        """
        root_dir: 应该是 'UNet/stage1_train' 的绝对路径或相对路径
        """
        self.root_dir = root_dir
        self.image_size = image_size
        self.mode = mode
        
        # 获取所有样本的 ID (就是那些乱码文件夹的名字)
        # 过滤掉非文件夹项 (比如 .DS_Store)
        self.image_ids = [x for x in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, x))]

        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            # 细胞分割通常不需要复杂的归一化，或者使用 ImageNet 的均值方差
            # transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        
        # --- 1. 智能读取原图 ---
        # 你的结构: /stage1_train/ID/images/xxx.png
        img_dir = os.path.join(self.root_dir, img_id, "images")
        
        # 自动获取 images 文件夹里的第一个文件名，防止文件名和 ID 不一致
        img_filename = os.listdir(img_dir)[0]
        img_path = os.path.join(img_dir, img_filename)
        
        # 读取图片并转 RGB (有的图可能是 RGBA 4通道，必须转 RGB)
        image = Image.open(img_path).convert("RGB")
        
        # --- 2. 读取并合并 Masks ---
        # 你的结构: /stage1_train/ID/masks/*.png
        mask_dir = os.path.join(self.root_dir, img_id, "masks")
        
        # 创建一个空的 mask (H, W)
        w, h = image.size
        combined_mask = np.zeros((h, w), dtype=np.uint8)
        
        # 如果是测试集可能没有 masks 文件夹，做个判断
        if os.path.exists(mask_dir):
            mask_files = os.listdir(mask_dir)
            for m_file in mask_files:
                if not m_file.endswith('.png'): continue
                
                m_path = os.path.join(mask_dir, m_file)
                # 读取单个细胞掩码
                m = np.array(Image.open(m_path).convert("L"))
                # 取最大值合并 (只要有一个mask在某像素有值，该像素就是前景)
                combined_mask = np.maximum(combined_mask, m)
        
        # --- 3. 预处理 ---
        # Resize Image
        image_tensor = self.transform(image)
        
        # Resize Mask (必须用最近邻插值 NEAREST，防止产生小数)
        mask_pil = Image.fromarray(combined_mask)
        mask_pil = mask_pil.resize((self.image_size, self.image_size), resample=Image.NEAREST)
        mask_np = np.array(mask_pil)
        
        # 二值化处理: 大于0的像素设为1 (前景), 0还是0 (背景)
        # 这一步很关键！因为 Resize 可能会引入杂色，且我们需要 0/1 标签
        mask_np = (mask_np > 0).astype(np.int64)
        
        mask_tensor = torch.from_numpy(mask_np).long()
        
        return image_tensor, mask_tensor

In [3]:
# 1. 设置路径
TRAIN_PATH = r'./stage1_train'  # 指向那个包含乱码文件夹的目录

# 2. 实例化
dataset = CellDataset(root_dir=TRAIN_PATH, image_size=512)

# 3. 检查一下是否读到了数据
print(f"Dataset size: {len(dataset)}") # 应该输出 670 左右 (DSB2018 训练集数量)

# 4.放入 DataLoader
dataloader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=0) 

Dataset size: 670


In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import wandb
import numpy as np
import math



# ==========================================
# 工具函数：计算指标
# ==========================================
def calculate_metrics(pred_tensor, target_tensor, smooth=1e-6):
    pred_class = torch.argmax(pred_tensor, dim=1)
    pred_flat = pred_class.view(-1)
    target_flat = target_tensor.view(-1)
    tp = (pred_flat * target_flat).sum().float()
    fp = (pred_flat * (1 - target_flat)).sum().float()
    fn = ((1 - pred_flat) * target_flat).sum().float()
    iou = (tp + smooth) / (tp + fp + fn + smooth)
    dice = (2 * tp + smooth) / (2 * tp + fp + fn + smooth)
    precision = (tp + smooth) / (tp + fp + smooth)
    recall = (tp + smooth) / (tp + fn + smooth)
    return iou.item(), dice.item(), precision.item(), recall.item()

# ==========================================
# 训练主程序
# ==========================================
def train():
    CONFIG = {
        "project_name": "Cell_Segmentation_Advanced",
        "epochs": 200,
        "batch_size": 4, 
        "accumulate_steps": 8, 
        "lr_max": 1e-3,
        "lr_min": 1e-6,
        "warmup_epochs": 5,
        "num_classes": 2,
        "image_size": 512,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "data_path": r"stage1_train" 
    }

    wandb.init(project=CONFIG["project_name"], config=CONFIG)
    
    # 准备数据
    full_dataset = CellDataset(root_dir=CONFIG['data_path'], image_size=CONFIG['image_size'])
    train_size = int(0.9 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0, pin_memory=True)
    
    # 模型与优化器
    model = UNet(3, CONFIG['num_classes']).to(CONFIG['device'])
    optimizer = optim.AdamW(model.parameters(), lr=CONFIG['lr_max'], weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    
    # 混合精度 Scaler
    scaler = torch.amp.GradScaler('cuda')

    # 学习率调度器
    def lr_lambda(epoch):
        if epoch < CONFIG['warmup_epochs']:
            return (epoch + 1) / CONFIG['warmup_epochs']
        else:
            progress = (epoch - CONFIG['warmup_epochs']) / (CONFIG['epochs'] - CONFIG['warmup_epochs'])
            cosine_decay = 0.5 * (1 + math.cos(math.pi * progress))
            return cosine_decay

    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

    print(f"🚀 Training on {CONFIG['device']}...")

    # --- 训练循环 ---
    for epoch in range(CONFIG['epochs']):
        model.train()
        epoch_loss = 0
        metrics_sum = np.zeros(4) # [iou, dice, precision, recall]
        optimizer.zero_grad() 
        
        for i, (images, masks) in enumerate(train_loader):
            images = images.to(CONFIG['device'])
            masks = masks.to(CONFIG['device'])
            
            # --- 修正：autocast 设备写错 ---
            # 原文是 'auda'，必须改为 'cuda'
            with torch.amp.autocast('cuda'): 
                outputs = model(images)
                loss = criterion(outputs, masks)
                loss = loss / CONFIG['accumulate_steps']

            scaler.scale(loss).backward()
            
            if (i + 1) % CONFIG['accumulate_steps'] == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
            
            epoch_loss += loss.item() * CONFIG['accumulate_steps']
            # 计算 Batch 指标
            batch_metrics = calculate_metrics(outputs, masks)
            metrics_sum += batch_metrics
            
            # 记录 Batch Loss
            wandb.log({"Batch/Train_Loss": loss.item() * CONFIG['accumulate_steps']})

        current_lr = optimizer.param_groups[0]['lr']
        scheduler.step()
        
        avg_train_loss = epoch_loss / len(train_loader)
        avg_train_metrics = metrics_sum / len(train_loader)

        # === 验证集 ===
        model.eval()
        val_loss = 0
        val_metrics_sum = np.zeros(4)
        log_images = []
        
        with torch.no_grad():
            for i, (images, masks) in enumerate(val_loader):
                images = images.to(CONFIG['device'])
                masks = masks.to(CONFIG['device'])
                
                with torch.amp.autocast('cuda'):
                    outputs = model(images)
                    loss = criterion(outputs, masks)
                
                val_loss += loss.item()
                val_metrics_sum += calculate_metrics(outputs, masks)
                
                # WandB 图片采样
                if i == 0:
                    pred_masks = torch.argmax(outputs, dim=1).cpu().numpy()
                    target_masks = masks.cpu().numpy()
                    input_imgs = images.cpu().numpy()
                    for k in range(min(2, images.size(0))):
                        img_vis = np.transpose(input_imgs[k], (1, 2, 0))
                        img_vis = (img_vis - img_vis.min()) / (img_vis.max() - img_vis.min())
                        log_images.append(wandb.Image(img_vis, masks={
                            "predictions": {"mask_data": pred_masks[k]},
                            "ground_truth": {"mask_data": target_masks[k]}
                        }, caption=f"Val_Epoch_{epoch+1}"))

        avg_val_loss = val_loss / len(val_loader)
        avg_val_metrics = val_metrics_sum / len(val_loader)

        # ===============================================
        # 🔑 关键修改：利用 "/" 对指标进行分组
        # ===============================================
        wandb.log({
            "epoch": epoch + 1,
            "Hyperparameters/Learning_Rate": current_lr,
            
            # 将 Loss 分组
            "Loss/Train": avg_train_loss,
            "Loss/Val": avg_val_loss,
            
            # 将 验证集指标 分组 (Val Metrics)
            # 这样它们在 WandB 界面中会出现在同一个 Section 下
            "Val_Metrics/IoU": avg_val_metrics[0],
            "Val_Metrics/Dice": avg_val_metrics[1],
            "Val_Metrics/Precision": avg_val_metrics[2],
            "Val_Metrics/Recall": avg_val_metrics[3],
            
            # 将 训练集指标 分组 (Train Metrics)
            "Train_Metrics/IoU": avg_train_metrics[0],
            "Train_Metrics/Dice": avg_train_metrics[1],
            "Train_Metrics/Precision": avg_train_metrics[2],
            "Train_Metrics/Recall": avg_train_metrics[3],

            "Visualization/Samples": log_images
        })
        
        print(f"Epoch [{epoch+1}/{CONFIG['epochs']}] LR: {current_lr:.6f} | Train Loss: {avg_train_loss:.4f} | Val IoU: {avg_val_metrics[0]:.4f}")
        
        torch.save(model.state_dict(), "best_resunet_3050ti.pth")

    wandb.finish()

train()

epoch,▁
lr,▁
train_batch_loss,███▆▆▅▄▄▅▄▅▄▃▄▄▃▃▄▃▂▃▄▂▃▂▂▂▂▂▂▂▂▁▂▂▁▁▂▁▂
train_loss,▁
val_dice,▁
val_iou,▁
val_loss,▁
epoch,1
lr,0.0002
train_batch_loss,0.25409
train_loss,0.38597


🚀 Training on cuda...
Epoch [1/200] LR: 0.000200 | Train Loss: 0.3613 | Val IoU: 0.5672
Epoch [2/200] LR: 0.000400 | Train Loss: 0.2095 | Val IoU: 0.6932
Epoch [3/200] LR: 0.000600 | Train Loss: 0.1698 | Val IoU: 0.7360
Epoch [4/200] LR: 0.000800 | Train Loss: 0.1395 | Val IoU: 0.7488
Epoch [5/200] LR: 0.001000 | Train Loss: 0.1192 | Val IoU: 0.7857
Epoch [6/200] LR: 0.001000 | Train Loss: 0.1021 | Val IoU: 0.7620
Epoch [7/200] LR: 0.001000 | Train Loss: 0.0999 | Val IoU: 0.7218
Epoch [8/200] LR: 0.001000 | Train Loss: 0.0937 | Val IoU: 0.7677
Epoch [9/200] LR: 0.000999 | Train Loss: 0.0871 | Val IoU: 0.7379
Epoch [10/200] LR: 0.000999 | Train Loss: 0.0884 | Val IoU: 0.7296
Epoch [11/200] LR: 0.000998 | Train Loss: 0.0853 | Val IoU: 0.7544
Epoch [12/200] LR: 0.000998 | Train Loss: 0.0812 | Val IoU: 0.7844
Epoch [13/200] LR: 0.000997 | Train Loss: 0.0805 | Val IoU: 0.7612
Epoch [14/200] LR: 0.000996 | Train Loss: 0.0787 | Val IoU: 0.7872
Epoch [15/200] LR: 0.000995 | Train Loss: 0.0759 

Batch/Train_Loss,▄▂▃█▆█▄▇▄▂▂▁▄▁▄▆▅▂▆▁▂▂▅▅▃▂▅▃▃▂▂▃▂▃▄▂▁▂▁▁
Hyperparameters/Learning_Rate,▄███████▇▇▇▇▇▇▇▆▆▆▆▆▆▆▆▅▅▄▄▄▄▃▂▂▂▂▂▂▁▁▁▁
Loss/Train,█▅▄▄▄▄▃▃▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
Loss/Val,█▃▂▂▂▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Train_Metrics/Dice,▁▄▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇█████████████
Train_Metrics/IoU,▁▂▂▂▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇██████
Train_Metrics/Precision,▁▂▂▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇█████████
Train_Metrics/Recall,▁▂▄▄▅▅▅▅▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▇▆▆▇▇▆▇▇▇███████
Val_Metrics/Dice,▅▃▅▁▆▆▇▇▇▇▇▆▇▆▆▇▇▇▅▆▇▇▇▇█▇▇▇█████▇▇█████
Val_Metrics/IoU,▁▆▆▇▇▇▇▇▇▇▇▇██▇██▇█▇█▇█▇███▇▇███████████
+3,...


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = UNet(3,2).to(device)
dummy_input = torch.randn(2, 3, 512, 512).to(device)

print("Testing UNet final version...")
output = model(dummy_input)

print(f"Input: {dummy_input.shape}")
print(f"Output: {output.shape}")

# 验证是否成功
assert output.shape == (2, 2, 512, 512), "Output shape mismatch!"
print("✅ Model passed shape check!")

In [ ]:


dummy_input = torch.randn(2, 3, 512, 512)
model = Encoder(ins=3)

print("正在测试模型流向...")
try:
    output ,f= model(dummy_input)
    print("\n✅ 模型运行成功!")
    print(f"输入尺寸: {dummy_input.shape}")
    print(f"输出尺寸: {output.shape}")
    
    # 验证 Layer 4 
    # 224 / 16 = 14
    stem_out = model.cbr(dummy_input)
    stem_out=model.m(stem_out)
    l1_out = model.layer1(stem_out)
    l2_out = model.layer2(l1_out)
    l3_out = model.layer3(l2_out)
    l4_out = model.layer4(l3_out)
    
    print("\n--- 内部尺寸检查 ---")
    print(f"Layer 1 Output: {l1_out.shape} ")
    print(f"Layer 2 Output: {l2_out.shape} ")
    print(f"Layer 3 Output: {l3_out.shape} ")
    print(f"Layer 4 Output: {l4_out.shape} ")
    
except Exception as e:
    print("\n❌ 发生错误:")
    print(e)

In [ ]:
# 3. Head (分类头)
print(l4_out.size(2))
l4_out = F.interpolate(l4_out, size=l4_out.size(2)*2, mode='bilinear', align_corners=True)
print(l4_out.shape)
filters=[]
for x in f:
    filters.append(x.size(1))
for x in filters:
    print(x)